# Statistical Validation Notebook (Target from Config)

This notebook runs model-level statistical tests for the currently active target in `configs/pipeline_config.yaml` (currently PHP).

Included blocks:
- ARIMA parameter significance (z-stats and p-values)
- VAR parameter significance (t-stats and p-values)
- Diebold-Mariano (DM) pairwise forecast comparison tests
- Residual diagnostics (Ljung-Box, ARCH-LM, Jarque-Bera)

All logic is modularized in `src/statistical_validation.py`.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')

from statistical_validation import (
    set_reproducible_seed,
    load_config,
    discover_forecasts,
    estimate_arima_param_significance,
    estimate_var_param_significance,
    build_default_dm_pairs,
    run_dm_comparisons,
    residual_diagnostics_table,
    garch_residual_diagnostics,
)

#### Interpretation
This initialization confirms packages and helper methods for statistical testing. No output table here means this cell is purely environment preparation for reproducible inference.

In [2]:
config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
print(f'Active target from config: {active_target}')

forecasts = discover_forecasts(config)
print('Forecast rows:', len(forecasts))
print('Models discovered:', sorted(forecasts['Model'].unique().tolist()))
display(forecasts.head())

Active target from config: PHP
Forecast rows: 38070
Models discovered: ['arima', 'baseline_ar1', 'baseline_mean', 'baseline_rw', 'hybrid_arima_mlp', 'hybrid_arima_svr', 'hybrid_var_mlp', 'hybrid_var_svr', 'var']


,Date,Actual,Forecast,Set,Pair,Model,Error,SE,AE
0,2022-12-30,-0.881993,-0.276428,val,USDPHP_RET,arima,-0.605565,0.366709,0.605565
1,2023-01-02,0.035926,0.163948,val,USDPHP_RET,arima,-0.128021,0.016389,0.128021
2,2023-01-03,0.143604,0.034660,val,USDPHP_RET,arima,0.108944,0.011869,0.108944
3,2023-01-04,0.241857,-0.029495,val,USDPHP_RET,arima,0.271352,0.073632,0.271352
4,2023-01-05,0.276976,-0.073465,val,USDPHP_RET,arima,0.350441,0.122809,0.350441


#### Interpretation
The printed target and loaded artifacts verify that evaluation files match the selected currency. Always check this before reading p-values or model rankings.

### What this notebook is doing

The cells below translate model output into formal statistical evidence. Parameter significance checks whether fitted coefficients are different from zero. DM tests compare forecast accuracy pairwise, while residual diagnostics check whether remaining errors still show autocorrelation, non-normality, or ARCH effects. Together, these tests justify why a linear first stage and a nonlinear residual learner are both needed.

## 1) ARIMA Parameter Significance

In [3]:
arima_sig = estimate_arima_param_significance(config)
arima_sig = arima_sig.sort_values(['pair', 'p_value'])
display(arima_sig)

arima_sig_summary = (
    arima_sig.groupby('pair', as_index=False)['significant_5pct']
    .mean()
    .rename(columns={'significant_5pct': 'share_significant_params'})
)
display(arima_sig_summary)

,pair,order,parameter,coef,std_err,z_stat,p_value,conf_low,conf_high,significant_5pct
5,CNYPHP_RET,"(0, 0, 1)",sigma2,0.208528,0.002727,76.456257,0.000000e+00,0.203182,0.213873,True
4,CNYPHP_RET,"(0, 0, 1)",ma.L1,-0.257972,0.010032,-25.715310,7.880922e-146,-0.277634,-0.238310,True
3,CNYPHP_RET,"(0, 0, 1)",const,0.005232,0.005901,0.886608,3.752901e-01,-0.006334,0.016797,False
11,HKDPHP_RET,"(0, 0, 1)",sigma2,0.188190,0.002423,77.654558,0.000000e+00,0.183440,0.192940,True
10,HKDPHP_RET,"(0, 0, 1)",ma.L1,-0.277919,0.009918,-28.022408,8.666658e-173,-0.297357,-0.258480,True
9,HKDPHP_RET,"(0, 0, 1)",const,0.005746,0.005437,1.056917,2.905495e-01,-0.004910,0.016402,False
8,JPYPHP_RET,"(0, 0, 1)",sigma2,0.506285,0.007639,66.277965,0.000000e+00,0.491313,0.521257,True
7,JPYPHP_RET,"(0, 0, 1)",ma.L1,-0.142025,0.012506,-11.356981,6.846984e-30,-0.166535,-0.117514,True
6,JPYPHP_RET,"(0, 0, 1)",const,-0.005008,0.010509,-0.476511,6.337104e-01,-0.025604,0.015589,False
14,SGDPHP_RET,"(0, 0, 1)",sigma2,0.204821,0.002675,76.572944,0.000000e+00,0.199579,0.210064,True


,pair,share_significant_params
0,CNYPHP_RET,0.666667
1,HKDPHP_RET,0.666667
2,JPYPHP_RET,0.666667
3,SGDPHP_RET,0.666667
4,USDPHP_RET,0.666667


#### Interpretation
ARIMA parameter significance indicates whether linear lag terms contribute predictive signal. Higher shares of significant coefficients support retaining the linear backbone in hybrid models.

## 2) VAR Parameter Significance

In [4]:
var_sig = estimate_var_param_significance(config)
var_sig = var_sig.sort_values(['equation', 'p_value'])
display(var_sig)

var_sig_summary = (
    var_sig.groupby('equation', as_index=False)['significant_5pct']
    .mean()
    .rename(columns={'significant_5pct': 'share_significant_params'})
)
display(var_sig_summary)

,equation,parameter,coef,std_err,t_stat,p_value,significant_5pct
13,CNYPHP_RET,L1.SGDPHP_RET,-0.199017,0.026800,-7.425915,1.120029e-13,True
11,CNYPHP_RET,L1.JPYPHP_RET,0.019758,0.014208,1.390636,1.643359e-01,False
9,CNYPHP_RET,L1.USDPHP_RET,-0.233056,0.226319,-1.029770,3.031181e-01,False
10,CNYPHP_RET,L1.CNYPHP_RET,0.031856,0.035360,0.900895,3.676443e-01,False
8,CNYPHP_RET,trend,0.000004,0.000008,0.445947,6.556354e-01,False
12,CNYPHP_RET,L1.HKDPHP_RET,0.066614,0.231299,0.287998,7.733483e-01,False
7,CNYPHP_RET,const,0.001557,0.015535,0.100211,9.201767e-01,False
27,HKDPHP_RET,L1.SGDPHP_RET,-0.277219,0.025373,-10.925620,8.694411e-28,True
24,HKDPHP_RET,L1.CNYPHP_RET,0.169471,0.033478,5.062233,4.143735e-07,True
22,HKDPHP_RET,trend,0.000010,0.000008,1.277597,2.013916e-01,False


,equation,share_significant_params
0,CNYPHP_RET,0.142857
1,HKDPHP_RET,0.285714
2,JPYPHP_RET,0.428571
3,SGDPHP_RET,0.428571
4,USDPHP_RET,0.285714


#### Interpretation
VAR significance summarizes cross-series lag interactions. Strong and frequent significance suggests multivariate spillovers are economically relevant for FX forecasting.

## 3) Diebold-Mariano Tests

In [5]:
models = sorted(forecasts['Model'].unique().tolist())
dm_pairs = build_default_dm_pairs(models)
print('DM comparisons configured:', len(dm_pairs))
for pair in dm_pairs[:10]:
    print('  ', pair)

dm_mse = run_dm_comparisons(forecasts, dm_pairs, criterion='mse', set_name='test')
dm_mae = run_dm_comparisons(forecasts, dm_pairs, criterion='mae', set_name='test')

display(dm_mse.sort_values(['Pair', 'p_value']))
display(dm_mae.sort_values(['Pair', 'p_value']))

DM comparisons configured: 22
   ('hybrid_arima_svr', 'arima')
   ('hybrid_arima_mlp', 'arima')
   ('hybrid_var_svr', 'var')
   ('hybrid_var_mlp', 'var')
   ('arima', 'baseline_ar1')
   ('arima', 'baseline_mean')
   ('arima', 'baseline_rw')
   ('hybrid_arima_mlp', 'baseline_ar1')
   ('hybrid_arima_mlp', 'baseline_mean')
   ('hybrid_arima_mlp', 'baseline_rw')


,Pair,Model_A,Model_B,Loss,Set,DM_stat,p_value,n_obs,A_better_than_B_5pct
4,CNYPHP_RET,arima,baseline_ar1,mse,test,-2.916155,0.003733,423,True
5,CNYPHP_RET,arima,baseline_mean,mse,test,-2.597348,0.009723,423,True
6,CNYPHP_RET,arima,baseline_rw,mse,test,-2.587261,0.010008,423,True
20,CNYPHP_RET,var,baseline_mean,mse,test,-2.194668,0.028732,423,True
21,CNYPHP_RET,var,baseline_rw,mse,test,-2.187641,0.029244,423,True
...,...,...,...,...,...,...,...,...,...
107,USDPHP_RET,var,baseline_ar1,mse,test,-1.896779,0.058540,423,False
91,USDPHP_RET,hybrid_var_mlp,var,mse,test,-1.861202,0.063411,423,False
104,USDPHP_RET,hybrid_var_svr,baseline_ar1,mse,test,-1.853863,0.064456,423,False
88,USDPHP_RET,hybrid_arima_svr,arima,mse,test,-1.833911,0.067371,423,False


,Pair,Model_A,Model_B,Loss,Set,DM_stat,p_value,n_obs,A_better_than_B_5pct
4,CNYPHP_RET,arima,baseline_ar1,mae,test,-3.614234,0.000338,423,True
7,CNYPHP_RET,hybrid_arima_mlp,baseline_ar1,mae,test,-2.750947,0.006198,423,True
9,CNYPHP_RET,hybrid_arima_mlp,baseline_rw,mae,test,-2.352573,0.019102,423,True
8,CNYPHP_RET,hybrid_arima_mlp,baseline_mean,mae,test,-2.341135,0.019690,423,True
1,CNYPHP_RET,hybrid_arima_mlp,arima,mae,test,-1.818846,0.069644,423,False
...,...,...,...,...,...,...,...,...,...
104,USDPHP_RET,hybrid_var_svr,baseline_ar1,mae,test,-1.709983,0.088004,423,False
90,USDPHP_RET,hybrid_var_svr,var,mae,test,-1.705222,0.088889,423,False
108,USDPHP_RET,var,baseline_mean,mae,test,-1.565537,0.118207,423,False
109,USDPHP_RET,var,baseline_rw,mae,test,-1.554818,0.120739,423,False


#### Interpretation
DM test matrices compare forecast errors model-by-model. Significant p-values imply meaningful predictive differences, while non-significance means models are statistically indistinguishable.

## 4) Residual Diagnostics

## 4.5) GARCH(1,1) Residual Volatility Comparison

This block fits a simple GARCH(1,1) on each model's test-set residuals. It is a compact way to compare how much conditional heteroskedasticity remains after the linear stage and after the hybrid correction stage.

In [6]:
diag = residual_diagnostics_table(forecasts, lags=10)
diag_test = diag[diag['Set'] == 'test'].copy()
display(diag_test.sort_values(['Pair', 'Model']))

reject_share = (
    diag_test.groupby('Model', as_index=False)[['LB_reject_5pct', 'ARCH_reject_5pct', 'JB_reject_5pct']]
    .mean()
    .sort_values('ARCH_reject_5pct', ascending=False)
)
display(reject_share)

,Pair,Model,Set,N,LB_pvalue,ARCHLM_pvalue,JB_pvalue,LB_reject_5pct,ARCH_reject_5pct,JB_reject_5pct
0,CNYPHP_RET,arima,test,423,1.562681e-03,2.365142e-08,2.380504e-215,True,True,True
2,CNYPHP_RET,baseline_ar1,test,423,3.980880e-05,2.983039e-08,1.723033e-215,True,True,True
4,CNYPHP_RET,baseline_mean,test,423,1.238208e-10,1.185839e-06,0.000000e+00,True,True,True
6,CNYPHP_RET,baseline_rw,test,423,1.238208e-10,1.383160e-06,0.000000e+00,True,True,True
8,CNYPHP_RET,hybrid_arima_mlp,test,423,1.860683e-03,2.176220e-03,9.359717e-99,True,True,True
10,CNYPHP_RET,hybrid_arima_svr,test,423,6.814171e-03,2.104276e-04,9.837237e-101,True,True,True
12,CNYPHP_RET,hybrid_var_mlp,test,423,2.168379e-05,8.263398e-04,6.944156e-93,True,True,True
14,CNYPHP_RET,hybrid_var_svr,test,423,2.096722e-04,2.991552e-03,1.880256e-90,True,True,True
16,CNYPHP_RET,var,test,423,6.199577e-05,3.855541e-08,4.517388e-183,True,True,True
18,HKDPHP_RET,arima,test,423,5.496068e-03,8.869360e-10,1.046298e-175,True,True,True


,Model,LB_reject_5pct,ARCH_reject_5pct,JB_reject_5pct
0,arima,1.0,1.0,1.0
1,baseline_ar1,1.0,1.0,1.0
2,baseline_mean,1.0,1.0,1.0
3,baseline_rw,1.0,1.0,1.0
4,hybrid_arima_mlp,1.0,1.0,1.0
8,var,1.0,1.0,1.0
6,hybrid_var_mlp,0.8,0.8,1.0
5,hybrid_arima_svr,1.0,0.6,1.0
7,hybrid_var_svr,0.8,0.4,1.0


#### Interpretation
Residual diagnostics combine autocorrelation, ARCH, and normality checks. Rejections indicate remaining structure in errors and justify nonlinear residual learners in hybrid setups.

In [7]:
out_dir = f'results/{active_target}/evaluation'
os.makedirs(out_dir, exist_ok=True)

arima_sig.to_csv(f'{out_dir}/arima_parameter_significance.csv', index=False)
var_sig.to_csv(f'{out_dir}/var_parameter_significance.csv', index=False)
dm_mse.to_csv(f'{out_dir}/dm_test_mse.csv', index=False)
dm_mae.to_csv(f'{out_dir}/dm_test_mae.csv', index=False)
diag_test.to_csv(f'{out_dir}/residual_diagnostics_test.csv', index=False)

print('Saved statistical outputs to', out_dir)

Saved statistical outputs to results/PHP/evaluation


#### Interpretation
This export confirmation means diagnostic tables were saved for reproducibility. Saved artifacts support downstream reporting and consistency checks across notebooks.

In [8]:
garch_df = garch_residual_diagnostics(forecasts)
if not garch_df.empty:
    display(garch_df.sort_values(['Pair', 'persistence']))
    
    garch_summary = (
        garch_df.groupby('Model', as_index=False)[['alpha1', 'beta1', 'persistence']]
        .mean()
        .sort_values('persistence', ascending=False)
    )
    display(garch_summary)
    garch_df.to_csv(f'{out_dir}/garch_residual_comparison.csv', index=False)
    garch_summary.to_csv(f'{out_dir}/garch_residual_comparison_summary.csv', index=False)
    print('Saved GARCH comparison outputs to', out_dir)

#### Interpretation
GARCH residual persistence compares volatility memory by model class. Higher persistence implies slower shock absorption and can guide risk-focused model choice.